<a href="https://colab.research.google.com/github/adrizooo/Marketing-Campaign-Data-Cleaning-Pipeline/blob/main/Marketing_Campaign_Data_Cleaning_%26_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Marketing Campaign Data Cleaning & Pipeline

This notebook demonstrates a professional, step-by-step pipeline to clean, validate, and engineer features from a messy marketing campaign dataset. We address typical real-world data issues such as structural schema inconsistencies, format conversion, logic anomalies, outliers, and text parsing.

### Core Stages:
1. **Environment Setup & Data Loading**
2. **Standardizing Column Headers & Removing Duplicates**
3. **Data Cleaning & Formatting**
4. **Logical Validation & Integrity Auditing**
5. **Outlier Detection & Capping**
6. **Feature Extraction & Engineering**

In [125]:
#The only 2 packages that we'll be using
import pandas as pd
import numpy as np

#Load the messy data
#df = data frame
df = pd.read_csv('marketing_campaign_data_messy.csv')

#Printing the messy dataset
print(f"Load Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Load Dataset: 2020 rows, 12 columns


## 🛠️ Step 1: Standardizing Column Headers & Removing Duplicates
We start by stripping trailing spaces, converting column names to lowercase, replacing spaces with underscores for ease of coding, and dropping duplicate columns (like the redundant `clicks` column).

In [126]:
# Clean the headers
# List all columns


print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')

print("FIX APPLIED")
print(df.columns.tolist ())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
FIX APPLIED
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


In [127]:
# Checking column (where it contains a dollar sign)
#Show first 3 rows
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

#Convert spend column into string & replace any characters that's
#not a digit, a dot or a minus with nothing
df['spend'] = df['spend'].astype(str).str.replace(r'\$','',regex=True)

#Convert string back to a numeric variable
df['spend'] = pd.to_numeric(df['spend'])

#See back the first 3 rows
print("FIX APPLIED")
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
FIX APPLIED
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


## 🧼 Step 2: Data Cleaning & Formatting
Here, we clean individual column values:
* Extract numeric representations from string-formatted currencies (removing the `$`).
* Standardize categorical values (such as correcting misspelled advertising platforms like `Facebok` and `Gogle`).
* Parse boolean states (converting messy values like `'Yes'`, `'Y'`, `'1'`, and `'0'` into true Boolean values).
* Convert date strings to standardized Pandas `datetime` formats.

In [128]:
#Display channels and different values that should be merged together
print(df['channel'].unique())

cleanup_map = {
    'Facebok':'Facebook',
    'Insta_gram':'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan #Handling the ghost value here
}

#Replace the values with cleanup map
df['channel'] = df['channel'].replace(cleanup_map)

print("FIX APPLIED")
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
FIX APPLIED
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [129]:
# Handling mixed Booleans (True/False)
#Cleaning up messy values
print(df['active'].unique())

bool_map = {
    'Yes': True,
    'Y': True,
    '1': True,
    1: True,
    'No': False,
    'N': False,
    '0': False,
    0: False
}

df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

#Check the values
print("FIX APPLIED")
print(df['active'].unique())


['Y' '0' 'No' 'True' 'Yes' '1' 'False']
FIX APPLIED
[ True False]


/tmp/ipykernel_413/2658367382.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)


In [130]:
#Changing the type for the start date

print(df['start_date'].dtype)

#Use pandas(pd) to convert the start date into a date time
# Coerce is used for formats that cannot convert into a date time
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] =pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("FIX APPLIED")
print(df['start_date'].dtype)

object
FIX APPLIED
datetime64[ns]


/tmp/ipykernel_413/449255127.py:8: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] =pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [131]:
#Remove duplicated 'Click' Column
df = df.loc[:,~df.columns.duplicated()]

In [132]:
#Checking
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


In [133]:
#View to see if clicks are more than impressions
#Define impossible and possible mask
#View campaign_id in first 3 rows
impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask,['campaign_id', 'clicks', 'impressions']].head(3))

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


## 🛡️ Step 3: Logical Validation & Integrity Auditing
Data pipelines should protect against impossible values. Here, we:
1. Audit whether clicks exceed total impressions.
2. Locate and correct "time-traveling" campaigns where `end_date` occurs before `start_date` by defaulting the campaign runtime to a logical 30 days.

In [134]:
#Make sure start date and end date are in chronological order

time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask,['campaign_id', 'start_date', 'end_date']].head(3))

#Assume that it's 30 days
df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print("FIX APPLIED")
print(df.loc[time_travel_mask,['campaign_id', 'start_date', 'end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
FIX APPLIED
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


## 📈 Step 4: Outlier Detection & Capping (IQR Method)
To prevent highly inflated spend metrics (like erroneous entries of $500,000) from skewing downstream analysis, we calculate an upper bound using the Interquartile Range (IQR) method and cap any extreme outliers to the 3x IQR boundary.

In [135]:
#Handling outliers using Interquartile range

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

#Check the instances if spend is higher than upper limit
outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask,['campaign_id', 'spend']].head(3))

print("FIX APPLIED")
#Capping the upper limit
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask,['campaign_id', 'spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
FIX APPLIED
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


## 🎨 Step 5: Feature Extraction & Engineering
To make our clean dataset richer for analysis, we extract the target campaign launch `season` from the structured text string embedded inside the `campaign_name` column using regular expressions.

In [136]:
#Create a new column based on an existing one

print(df['campaign_name'].head(3))

#Extracting from the first underscore and the second underscore
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print("FIX APPLIED")
print(df[['campaign_name', 'season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
FIX APPLIED
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [137]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter
